# Module 09: Statistical Visualization with Matplotlib
## Visualizing CHIRPS Rainfall Statistics for Ethiopia

In this module, you will learn to create statistical visualizations using real CHIRPS rainfall data (1981-2022) for Ethiopia. Topics include box plots, violin plots, error bars, confidence intervals, and distribution plots.

## Setup

Load required libraries and the prepared CHIRPS datasets.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

In [ ]:
DATA_DIR = '../datasets'
CHIRPS_PATH = '../chirps_1981_2022.nc'

ds = xr.open_dataset(CHIRPS_PATH, engine='netcdf4')
clim = xr.open_dataset(f'{DATA_DIR}/chirps_climatology.nc', engine='netcdf4')
annual = xr.open_dataset(f'{DATA_DIR}/chirps_annual.nc', engine='netcdf4')
regional = pd.read_csv(f'{DATA_DIR}/ethiopia_regional_rainfall.csv', parse_dates=['time'])
point_df = pd.read_csv(f'{DATA_DIR}/addis_ababa_rainfall.csv', parse_dates=['time'])

print(f'Full dataset: {ds.precip.shape}')
print(f'Climatology: {clim.precip.shape}')
print(f'Annual: {annual.precip.shape}')
print(f'Regional data: {regional.shape}')
print(f'Point data: {point_df.shape}')

## 1. Box Plots: Rainfall Distribution by Month

Box plots show the distribution of a dataset through quartiles. We'll visualize CHIRPS rainfall distribution for each month across all years (1981-2022) using data from Addis Ababa.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

months_data = [point_df[point_df['time'].dt.month == m]['precip'].values for m in range(1, 13)]

bp = ax.boxplot(months_data, patch_artist=True, widths=0.6,
                medianprops={'color': 'black', 'linewidth': 2})

colors = plt.cm.viridis(np.linspace(0.2, 0.8, 12))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('Monthly Rainfall Distribution at Addis Ababa (1981-2022)', fontsize=14, fontweight='bold')
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### Box Plot: Annual Rainfall Distribution by Decade

Compare annual rainfall totals across decades to identify trends in distribution.

In [ ]:
annual_point = ds.precip.sel(longitude=38.75, latitude=9.03, method='nearest').to_dataframe().reset_index()
annual_point['year'] = annual_point['time'].dt.year
annual_point['decade'] = (annual_point['year'] // 10) * 10

decades = sorted(annual_point['decade'].unique())
decade_data = [annual_point[annual_point['decade'] == d]['precip'].values for d in decades]

fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot(decade_data, patch_artist=True, widths=0.5,
                medianprops={'color': 'red', 'linewidth': 2})

colors_dec = plt.cm.Set2(np.linspace(0, 1, len(decades)))
for patch, color in zip(bp['boxes'], colors_dec):
    patch.set_facecolor(color)

ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Annual Precipitation (mm/year)', fontsize=12)
ax.set_title('Annual Rainfall Distribution by Decade', fontsize=14, fontweight='bold')
ax.set_xticklabels([str(d) for d in decades])
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 2. Violin Plots: Rainfall Distribution by Season

Violin plots combine box plots with KDE. They show the probability density of the data. We'll analyze rainfall by season across Ethiopia.

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'DJF (Dry)'
    elif month in [3, 4, 5]:
        return 'MAM (Long Rain)'
    elif month in [6, 7, 8]:
        return 'JJA (Main Rain)'
    else:
        return 'SON (Short Rain)'

regional['season'] = regional['time'].dt.month.map(get_season)
season_order = ['DJF (Dry)', 'MAM (Long Rain)', 'JJA (Main Rain)', 'SON (Short Rain)']

fig, ax = plt.subplots(figsize=(12, 6))

season_data = [regional[regional['season'] == s]['precip'].dropna().values for s in season_order]

vp = ax.violinplot(season_data, positions=[1, 2, 3, 4], showmeans=True, showmedians=True,
                   quantiles=[[0.25, 0.75]] * 4)

colors_v = ['#e74c3c', '#f39c12', '#27ae60', '#3498db']
for i, (pc, color) in enumerate(zip(vp['bodies'], colors_v)):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
    pc.set_edgecolor('black')

for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians', 'cmeans'):
    if partname in vp:
        vp[partname].set_edgecolor('black')
        vp[partname].set_linewidth(1.5)

ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('Rainfall Distribution by Season (All Ethiopian Regions)', fontsize=14, fontweight='bold')
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(season_order, fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 3. Error Bars: Mean ± Standard Deviation

Error bars indicate the variability of data. We'll plot mean monthly rainfall with ±1σ error bars for selected Ethiopian regions.

In [ ]:
monthly_stats = regional.groupby([regional['region'], regional['time'].dt.month])['precip'].agg(['mean', 'std', 'count']).reset_index()

selected_regions = ['Tigray', 'Amhara', 'Oromia', 'SNNPR', 'Somali']
region_colors = {'Tigray': '#e74c3c', 'Amhara': '#3498db', 'Oromia': '#2ecc71', 'SNNPR': '#f39c12', 'Somali': '#9b59b6'}

months_idx = np.arange(1, 13)

fig, ax = plt.subplots(figsize=(14, 7))

for region in selected_regions:
    rdata = monthly_stats[monthly_stats['region'] == region]
    ax.errorbar(months_idx, rdata['mean'], yerr=rdata['std'],
                fmt='o-', capsize=4, capthick=1.5, elinewidth=1.5,
                label=region, color=region_colors[region], markersize=6,
                markeredgecolor='black', markeredgewidth=0.5)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_title('Mean Monthly Rainfall ± 1σ for Ethiopian Regions (1981-2022)', fontsize=14, fontweight='bold')
ax.set_xticks(months_idx)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 4. Confidence Intervals

We compute 95% confidence intervals for mean monthly rainfall using the bootstrap method. This shows the uncertainty in our estimate of the population mean.

In [ ]:
def bootstrap_ci(data, n_bootstrap=10000, ci=95):
    data = np.array(data)[~np.isnan(data)]
    if len(data) < 2:
        return np.nan, np.nan
    means = np.zeros(n_bootstrap)
    n = len(data)
    for i in range(n_bootstrap):
        sample = np.random.choice(data, size=n, replace=True)
        means[i] = np.mean(sample)
    alpha = (100 - ci) / 2
    lower = np.percentile(means, alpha)
    upper = np.percentile(means, 100 - alpha)
    return lower, upper

regions_list = regional['region'].unique()
ci_results = []

for region in regions_list:
    for month in range(1, 13):
        mask = (regional['region'] == region) & (regional['time'].dt.month == month)
        d = regional.loc[mask, 'precip'].values
        lower, upper = bootstrap_ci(d)
        ci_results.append({'region': region, 'month': month,
                          'mean': np.nanmean(d), 'ci_lower': lower, 'ci_upper': upper})

ci_df = pd.DataFrame(ci_results)
print(f'Computed bootstrap CIs for {len(ci_df)} region-month combinations')
ci_df.head()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, region in enumerate(selected_regions[:6]):
    ax = axes[idx]
    rdata = ci_df[ci_df['region'] == region]

    ax.fill_between(rdata['month'], rdata['ci_lower'], rdata['ci_upper'],
                    alpha=0.3, color='steelblue', label='95% CI')
    ax.plot(rdata['month'], rdata['mean'], 'o-', color='darkblue',
            linewidth=2, markersize=5, label='Mean')

    ax.set_title(region, fontsize=12, fontweight='bold')
    ax.set_xlabel('Month', fontsize=10)
    ax.set_ylabel('Precip (mm)', fontsize=10)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=8)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

fig.suptitle('Monthly Mean Rainfall with 95% Bootstrap Confidence Intervals',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Distribution Plots: Histograms with KDE

Filled histograms with Kernel Density Estimation (KDE) overlays reveal the shape of rainfall distributions for different regions.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

hist_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for idx, region in enumerate(selected_regions[:6]):
    ax = axes[idx]
    rdata = regional[regional['region'] == region]['precip'].dropna().values

    ax.hist(rdata, bins=40, density=True, alpha=0.6,
            color=hist_colors[idx], edgecolor='black', linewidth=0.5,
            label='Histogram')

    x_vals = np.linspace(rdata.min(), rdata.max(), 200)
    kde = stats.gaussian_kde(rdata)
    ax.plot(x_vals, kde(x_vals), 'r-', linewidth=2.5, label='KDE')

    mean_val = np.mean(rdata)
    median_val = np.median(rdata)
    ax.axvline(mean_val, color='darkblue', linestyle='--', linewidth=2, label=f'Mean={mean_val:.1f}')
    ax.axvline(median_val, color='darkgreen', linestyle=':', linewidth=2, label=f'Median={median_val:.1f}')

    ax.set_title(region, fontsize=12, fontweight='bold')
    ax.set_xlabel('Precipitation (mm/month)', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(True, alpha=0.3)

fig.suptitle('Rainfall Distribution: Histograms with KDE Overlay',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Cumulative Distribution Functions (CDF)

CDFs show the probability that rainfall is ≤ a given value, useful for drought/flood threshold analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for idx, region in enumerate(selected_regions):
    rdata = regional[regional['region'] == region]['precip'].dropna().values
    sorted_data = np.sort(rdata)
    cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
    ax.plot(sorted_data, cdf, linewidth=2, label=region, color=list(region_colors.values())[idx])
    ax.fill_between(sorted_data, 0, cdf, alpha=0.1, color=list(region_colors.values())[idx])

ax.axhline(0.25, color='gray', linestyle='--', alpha=0.5, label='25th/75th pctl')
ax.axhline(0.75, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0.5, color='black', linestyle='-', alpha=0.3, label='Median')

ax.set_xlabel('Precipitation (mm/month)', fontsize=12)
ax.set_ylabel('Cumulative Probability', fontsize=12)
ax.set_title('Cumulative Distribution Functions of Monthly Rainfall', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## Exercises

1. **Regional Box Plot Comparison**: Create a grouped box plot comparing July rainfall across all 10 Ethiopian regions.
2. **Seasonal Violin Plot**: Generate violin plots for each season separately for the highland (Tigray, Amhara) vs lowland (Somali, Afar) regions.
3. **Confidence Interval by Year**: Compute 95% CIs for annual rainfall totals per region and plot with error bars.
4. **Multi-panel Distribution**: Create a 5×2 grid of histograms with KDE for all 10 Ethiopian regions, using consistent x-axis limits.
5. **Statistical Report**: Compute skewness, kurtosis, and the 90th percentile of rainfall for each region and visualize as a bar chart.

### Mini-Project: Ethiopian Rainfall Variability Analysis

Create a comprehensive statistical analysis report with:
- A figure with 4 subplots showing box plot, violin plot, error bar plot, and CDF for a region of your choice
- Bootstrap confidence intervals comparing two contrasting regions
- A summary table of statistical moments (mean, std, skew, kurtosis) for all regions
- Write a short interpretation of rainfall variability patterns across Ethiopia